# 02 · Baseline CNN (Sprint 1 rubric: working PyTorch CNN + baseline results)

Architecture: 3 × (Conv3×3 → ReLU → MaxPool2) with 32 → 64 → 128 filters, then Flatten → FC256 → ReLU → Dropout(0.5) → FC10.
Input: 64×64 RGB crop · Output: 10 logits (softmax applied at inference).

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt

# Works whether the notebook is opened from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
print("project root:", ROOT)

## 1 · Verify shapes and parameter count

In [ ]:
import torch
from saferoute.vision.baseline_cnn import BaselineSignCNN

print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
model = BaselineSignCNN()
x = torch.randn(4, 3, 64, 64)
for name, layer in [("features", model.features), ("classifier", model.classifier)]:
    x = layer(x)
    print(f"{name:10s} -> {tuple(x.shape)}")
print(f"trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(model)

## 2 · Short live training demo (2 epochs, does NOT overwrite the saved model)

In [ ]:
import train_baseline as tb
from saferoute.vision.datasets import build_loaders

device = "cuda" if torch.cuda.is_available() else "cpu"
train_dl, val_dl, _, class_names = build_loaders(ROOT / "data/processed/crops",
                                                 classes_yaml=ROOT / "configs/classes.yaml", num_workers=2)
demo = BaselineSignCNN(num_classes=len(class_names)).to(device)
opt = torch.optim.Adam(demo.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()
for ep in range(1, 3):
    tl, ta = tb.run_epoch(demo, train_dl, loss_fn, device, opt)
    vl, va = tb.run_epoch(demo, val_dl, loss_fn, device)
    print(f"demo epoch {ep}: train loss/acc {tl:.3f}/{ta:.3f} | val loss/acc {vl:.3f}/{va:.3f}")

## 3 · Full 20-epoch training run: loss and accuracy curves
(from `scripts/train_baseline.py`, saved in `reports/baseline_history.json`)

In [ ]:
hist = json.loads((ROOT / "reports/baseline_history.json").read_text())
h = hist["history"]
ep = [r["epoch"] for r in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [r["train_loss"] for r in h], label="train"); ax[0].plot(ep, [r["val_loss"] for r in h], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(ep, [r["train_acc"] for r in h], label="train"); ax[1].plot(ep, [r["val_acc"] for r in h], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"best val acc: {hist['best_val_acc']:.4f} | test acc: {hist['test_acc_last_epoch']:.4f}")

## 4 · Test-set evaluation of the saved model: per-class accuracy and confusion matrix

In [ ]:
from torch.utils.data import DataLoader
from saferoute.vision.datasets import OrderedImageFolder, build_transforms, class_names_from_yaml

names = class_names_from_yaml(ROOT / "configs/classes.yaml")
test_ds = OrderedImageFolder(ROOT / "data/processed/crops/test", names, build_transforms(64))
ckpt = torch.load(ROOT / "models/baseline_cnn.pt", map_location="cpu")
net = BaselineSignCNN(num_classes=len(ckpt["classes"]))
net.load_state_dict(ckpt["state_dict"]); net.eval()

cm = np.zeros((len(names), len(names)), dtype=int)
with torch.no_grad():
    for xb, yb in DataLoader(test_ds, batch_size=256):
        for t, p in zip(yb.tolist(), net(xb).argmax(1).tolist()):
            cm[t, p] += 1
print(f"test accuracy: {np.trace(cm) / cm.sum():.4f}  (n={cm.sum()})\n")
for i, n in enumerate(names):
    print(f"{n:20s} {cm[i, i] / cm[i].sum():.3f}  (n={cm[i].sum()})")

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Baseline CNN, GTSRB test split")
plt.colorbar(im); plt.tight_layout()
fig.savefig(ROOT / "reports/baseline_confusion_matrix.png", dpi=150); plt.show()

## 5 · Same model through ONNX Runtime (what the FastAPI service uses)

In [ ]:
from saferoute.vision.classifier_engine import ClassifierEngine

eng = ClassifierEngine(ROOT / "models/baseline_cnn.onnx", names)
print("execution provider:", eng.provider, "| sha256:", eng.model_hash[:16], "...")
for cls in ["STOP", "SPEED_LIMIT_60", "SCHOOL_ZONE"]:
    sample = next((ROOT / "data/processed/crops/test" / cls).iterdir())
    out = eng.predict(sample.read_bytes())
    print(f"true={cls:16s} predicted={out['label']:16s} conf={out['confidence']:.3f}  {out['inference_ms']:.2f} ms")